# QLoRA Fine-tuning

`1. VLM 학습 데이터 준비.ipynb`에서 만든 데이터셋으로 **Qwen2-VL-2B-Instruct**를 QLoRA(4bit 양자화 + LoRA)로 fine-tuning합니다.

> ⚠️ RunPod GPU 인스턴스(24GB VRAM급 이상 권장)에서 실행하세요. CPU 환경에서는 4bit 양자화 로딩이 정상 동작하지 않습니다.

## 사전 준비
```bash
uv sync
huggingface-cli login   # Qwen2-VL 다운로드용 (필요 시)
```


In [ ]:
import torch
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2VLForConditionalGeneration,
)
from peft import LoraConfig, get_peft_model

import vlm_finetune_utils as vu

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"


## 1. 4bit 양자화로 base 모델 로드
QLoRA 방식대로, 원본 가중치는 4bit로 양자화해서 올리고 LoRA(A, B) 파라미터만 원래 정밀도(bf16)로 학습합니다.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)


## 2. LoRA 설정
언어 모델의 attention/MLP 프로젝션 레이어에 저랭크 어댑터(A, B)를 붙입니다. `r`(랭크)이 클수록 표현력은 늘지만 학습 파라미터도 늘어납니다.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable, total = vu.count_trainable_params(model)
print(f"학습 파라미터: {trainable:,} / 전체 {total:,} ({trainable / total:.4%})")


## 3. 데이터셋 로드 및 전처리
`to_chat_messages`로 각 레코드를 processor의 chat template 입력 형태로 바꿉니다.

In [ ]:
train_records = vu.load_jsonl("data/train/annotations.jsonl")
val_records = vu.load_jsonl("data/val/annotations.jsonl")

train_messages = [vu.to_chat_messages(r, "data/train") for r in train_records]
train_messages[0]


In [ ]:
from qwen_vl_utils import process_vision_info


def collate_fn(batch_messages):
    texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch_messages
    ]
    image_inputs, _ = process_vision_info(
        [msg for messages in batch_messages for msg in messages]
    )
    batch = processor(
        text=texts, images=image_inputs, return_tensors="pt", padding=True
    )
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    return batch


## 4. 학습
토이 데이터셋(50장)이라 몇 epoch만으로도 패턴이 학습되는지 확인할 수 있습니다.

In [ ]:
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

loader = DataLoader(train_messages, batch_size=2, shuffle=True, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-4
)
num_epochs = 3
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=5, num_training_steps=num_epochs * len(loader)
)


In [ ]:
model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in loader:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    print(f"epoch {epoch + 1}/{num_epochs} - loss {total_loss / len(loader):.4f}")


## 5. 어댑터 저장
학습된 LoRA 가중치(A, B)만 저장합니다 — 원본 모델은 그대로 두므로 결과물이 작습니다.

In [ ]:
ADAPTER_DIR = "outputs/qlora-defect-adapter"
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print("저장 완료:", ADAPTER_DIR)


## LoRA와의 차이 정리
- 이 노트북은 `BitsAndBytesConfig(load_in_4bit=True)`로 **QLoRA**를 사용했습니다.
- 순수 LoRA로 하려면 `quantization_config` 없이 `torch_dtype=torch.bfloat16`으로만 모델을 로드하면 됩니다 — VRAM이 더 필요한 대신 양자화 오차가 없어 약간 더 안정적입니다.
- 다음 노트북에서 이 어댑터를 base 모델과 비교 평가합니다.